# Instamart Customer Intelligence & Support Copilot
End-to-end walk-through. Every cell calls the same modules as `run_project.py`, so this notebook and the scripts cannot drift apart. All data is synthetic, and all language-model calls use the supplied offline `mock_llm.py`, so no API key is needed.

Install the project requirements and run the cells from top to bottom. Jupyter itself is not a project requirement: `pip install notebook` to open this file.

In [ ]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "generate_data.py").exists() else Path.cwd().parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root found:", (ROOT / "generate_data.py").exists())

## 1. Generate the raw data
The generator is deterministic, so rerunning it recreates identical exports.

In [ ]:
print(subprocess.run([sys.executable, "generate_data.py"], check=True, capture_output=True, text=True, cwd=ROOT).stdout)
print(sorted(p.name for p in (ROOT / "data" / "raw").iterdir()))

## 2. Part A: audit, clean, store
Raw files are read as literal text for the audit, so blanks and `N/A` stay visible. Each cleaning rule records how many rows or values it changed.

In [ ]:
from src import data_pipeline, analytics
from src.validate_checkpoints import validate

part_a = data_pipeline.run()
display(pd.Series(part_a["audit"]["figures"], name="audit figure").to_frame())
display(pd.Series({**part_a["customer_rules"], **part_a["order_rules"]}, name="changed").to_frame())
validate()
print("ALL CHECKPOINTS PASSED")

## 3. Part A: SQL and pandas analysis
The seven queries live in `sql/queries.sql`. The pivot table independently rebuilds the revenue total.

In [ ]:
results = analytics.run_queries()
display(results["Q2"].head(5))
display(results["Q6"].head(3))
display(results["Q7"].tail(2))
summary = analytics.run()
print("SQL total INR:", summary["cross_check"]["sql_total_inr"], "| pivot total INR:", summary["cross_check"]["pivot_grand_total_inr"])
display(pd.DataFrame(summary["cross_check"]["per_city"]))
print({k: v for k, v in summary["exploratory"].items() if k != "correlation"})
display(pd.DataFrame(summary["exploratory"]["correlation"]))

## 4. Part B: segments
K-means runs on standardised RFM. K=4 has the sharpest elbow and the best silhouette. Cluster numbers are only labels with no ordering.

In [ ]:
from src import customer_models
part_b = customer_models.run()
segments, lapse = part_b["segments"], part_b["lapse"]
display(pd.DataFrame(segments["kmeans"]))
display(pd.DataFrame(segments["profile"]).drop(columns="action"))
print("PCA explained variance:", segments["pca_explained_variance_ratio"], "total:", segments["pca_total"])

## 5. Part B: lapse model
Features use only orders before 2025-09-01, and September defines the label. Scaling and encoding sit inside a Pipeline fitted on training rows only. The test set is scored only after tuning finishes.

In [ ]:
print(lapse["cohort"])
print("Baseline:", lapse["baseline_always_not_lapsed"])
rows = {name: {"best_cv_f1": lapse[name]["best_cv_f1"], **lapse[name]["test"]} for name in ("logistic_regression", "random_forest")}
rows["logistic_balanced"] = lapse["class_balance"]["balanced"]
display(pd.DataFrame(rows).T)
print("Selected:", lapse["selected_model"], "| top features:", lapse["top_rf_features"])
display(pd.DataFrame(lapse["segment_lapse"]))

The retention list below is illustrative and not a leakage-free evaluation, because the Premium flag is a post-hoc whole-window label that includes September.

In [ ]:
display(pd.read_csv(ROOT / "outputs" / "retention_list.csv"))
print("Truly lapsed in list:", lapse["retention_list"]["truly_lapsed"], "of", lapse["retention_list"]["listed"])

## 6. Part C: ticket triage
Each ticket passes input guardrails, ordered redaction, a budget check, retries, output validation and routing. Anything doubtful goes to a human. The retry lines below are the pipeline's own log.

In [ ]:
from src import triage
part_c = triage.run_all()
print("Redaction:", part_c["redaction"])
display(pd.DataFrame({k: part_c[k] for k in ("zero_shot", "few_shot")}).drop(index=["status_counts", "route_counts"]))
display(pd.DataFrame(part_c["zero_shot_confusion_true_by_predicted"]).T)
print("Budget demo:", part_c["budget_demo"]["budget_exhausted"], "tickets exhausted at", part_c["budget_demo"]["token_budget"], "tokens")
display(pd.read_csv(ROOT / "outputs" / "triage_results.csv", keep_default_na=False).head(8))

## 7. Part D: policy assistant
Chunks are 40 words with a 10-word overlap, ranked by TF-IDF cosine similarity. The assistant answers only from the retrieved context.

In [ ]:
from src import policy_rag
part_d_rag = policy_rag.run()
print(policy_rag.answer("Who must approve a refund above INR 1000?")["answer"])
print(policy_rag.answer("Who won the cricket world cup final?")["answer"])
for name in ("overlap_10", "overlap_0"):
    e = part_d_rag[name]
    print(name, "chunks", e["chunks"], "| doc", e["doc_in_top3"], "| fact", e["fact_in_answer"], "| fallback", e["fallback"])
print("Broken by zero overlap:", [q["question"] for q in part_d_rag["overlap_0"]["failed_questions"]])

## 8. Part D: refund agent
The agent runs the seven scenarios, then the stop-safely and crash-and-resume demonstrations. The ledger is a local audit file, and no money moves.

In [ ]:
from src import refund_agent
part_d_agent = refund_agent.run()
display(pd.DataFrame(part_d_agent["scenarios"]))
demos = part_d_agent["demonstrations"]
print("token_limit=200:", demos["token_limit_200"]["status"])
print("time_limit_seconds=0:", demos["time_limit_0"]["status"])
print("crash and resume:", demos["crash_resume"]["tools_all"], demos["crash_resume"]["status"], demos["crash_resume"]["refund_inr"])

## 9. Generated artefacts

In [ ]:
for folder in ("data/clean", "outputs", "state"):
    print(folder, sorted(p.name for p in (ROOT / folder).iterdir() if p.is_file()))
print("Database present:", (ROOT / "data" / "instamart.db").exists())